# CSCE 704 — Consolidated Notebook
**Project 1** (Data Prep) -> **Project 2** (Per-Filetype RF Training) -> **Project 3** (Global Ensemble + Challenge Eval)

## 1. Environment Setup

In [ ]:
!git clone https://github.com/FutureComputing4AI/EMBER2024.git
%cd EMBER2024
!pip install .

# Create a directory to store the data
!mkdir /content/ember_data

# To resolve the dependency error I faced during the run.
!pip uninstall -y signify
!pip install signify==0.8.1

## 2. Imports

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import joblib
import thrember

## 3. Data Preparation

Run once to download raw EMBER data and create chunked `.npz` files. Skip if chunks already exist on Drive.

### 3.1 PDF / APK / ELF — Vectorized Feature Chunking

In [ ]:
import os
import numpy as np
import thrember

def create_and_save_chunks(
    dataset_path: str,
    output_dir: str,
    subset: str = "train",
    num_chunks: int = 4,
    file_prefix: str = "ember_chunk",
    random_state: int = 42
):
    """
    Loads vectorized EMBER2024 features, splits them into disjoint chunks,
    and saves each chunk as a .npz file.

    Parameters:
    - dataset_path (str): Path where EMBER2024 dataset is stored
    - output_dir (str): Directory to save chunk files
    - subset (str): "train", "test", or "challenge"
    - num_chunks (int): Number of chunks to split into
    - file_prefix (str): Prefix for saved files
    - random_state (int): Seed for reproducibility
    """

    os.makedirs(output_dir, exist_ok=True)

    print(f"Loading {subset} data from {dataset_path}...")
    X, y = thrember.read_vectorized_features(dataset_path, subset=subset)

    print(f"Total samples: {X.shape[0]}")
    print("Shuffling indices...")

    rng = np.random.default_rng(seed=random_state)
    indices = rng.permutation(X.shape[0])

    print(f"Splitting into {num_chunks} chunks...")
    chunks = np.array_split(indices, num_chunks)

    for i, chunk_idx in enumerate(chunks):
        X_chunk = X[chunk_idx]
        y_chunk = y[chunk_idx]

        file_path = os.path.join(output_dir, f"{file_prefix}_{subset}_{i}.npz")

        np.savez_compressed(file_path, X=X_chunk, y=y_chunk)

        print(f"Saved: {file_path} | Shape: {X_chunk.shape}")

    print("All chunks saved successfully.")


filetype_list = ["PDF", "APK", "ELF"]

for filetype in filetype_list:

  thrember.download_dataset("/content/ember_data", file_type=filetype)
  thrember.create_vectorized_features('/content/ember_data')

  create_and_save_chunks(
      dataset_path="/content/ember_data",
      output_dir="/content/drive/MyDrive/ember_chunks",
      subset="train",
      num_chunks=4,
      file_prefix=f"ember_{filetype.lower()}_train"
  )

  X_test, y_test = thrember.read_vectorized_features(
      "/content/ember_data",
      subset="test"
  )

  np.savez_compressed(
      f"/content/drive/MyDrive/ember_{filetype.lower()}_test.npz",
      X=X_test,
      y=y_test
  )

### 3.2 Win32 / Win64 / Dot_Net — Memory-mapped Chunking

In [ ]:
import os
import numpy as np


def chunk_dat_with_memmap(
    data_dir: str,
    output_dir: str,
    subset: str = "train",
    num_features: int = 2568,
    chunk_size: int = 100000,
    file_prefix: str = "ember_chunk",
    dtype_X=np.float32,
    dtype_y=np.int32
):
    """
    Memory-efficient chunking of EMBER .dat files using numpy memmap.

    Parameters:
    - data_dir (str): Directory containing X_*.dat and y_*.dat
    - output_dir (str): Directory to save .npz chunk files
    - subset (str): "train", "test", or "challenge"
    - num_features (int): Number of features (default: 2568 for EMBER v3)
    - chunk_size (int): Number of samples per chunk
    - file_prefix (str): Prefix for saved files
    - dtype_X: Data type of X (usually float32)
    - dtype_y: Data type of y (usually int32)
    """

    os.makedirs(output_dir, exist_ok=True)

    X_path = os.path.join(data_dir, f"X_{subset}.dat")
    y_path = os.path.join(data_dir, f"y_{subset}.dat")

    if not os.path.exists(X_path) or not os.path.exists(y_path):
        raise FileNotFoundError(f"Missing .dat files for subset: {subset}")

    print(f"Opening memmap for {subset}...")

    # Memory-map raw data
    X_mem = np.memmap(X_path, dtype=dtype_X, mode='r')
    y_mem = np.memmap(y_path, dtype=dtype_y, mode='r')

    # Infer number of samples
    num_samples = X_mem.shape[0] // num_features

    if X_mem.shape[0] % num_features != 0:
        raise ValueError("X data size is not divisible by num_features")

    # Reshape X properly
    X_mem = X_mem.reshape(num_samples, num_features)

    print(f"Total samples: {num_samples}")
    print(f"Chunk size: {chunk_size}")
    print(f"Number of chunks: {(num_samples + chunk_size - 1) // chunk_size}")

    # Chunking loop
    chunk_id = 0

    for start in range(0, num_samples, chunk_size):
        end = min(start + chunk_size, num_samples)

        X_chunk = X_mem[start:end]
        y_chunk = y_mem[start:end]

        save_path = os.path.join(
            output_dir,
            f"{file_prefix}_{subset}_{chunk_id}.npz"
        )

        np.savez_compressed(save_path, X=X_chunk, y=y_chunk)

        print(f"Saved: {save_path} | Shape: {X_chunk.shape}")

        chunk_id += 1

    print("All chunks saved successfully.")

filetype_list = ["Win32", "Win64", "Dot_Net"]

for filetype in filetype_list:
    chunk_dat_with_memmap(
        data_dir="/content/ember_data",
        output_dir="/content/drive/MyDrive/ember_chunks",
        subset="train",
        num_features=2568,
        chunk_size=100000,
        file_prefix=f"{filetype.lower()}"
    )

    X_test, y_test = thrember.read_vectorized_features(
      "/content/ember_data",
      subset="test"
    )

    np.savez_compressed(
      f"/content/drive/MyDrive/ember_{filetype.lower()}_test.npz",
      X=X_test,
      y=y_test
    )

## 4. Helper Functions

In [ ]:
def load_chunks(file_paths):
    X_list, y_list = [], []

    for path in file_paths:
        data = np.load(path)
        X_list.append(data["X"])
        y_list.append(data["y"])

    return np.vstack(X_list), np.hstack(y_list)

In [ ]:
def train_rf(
    X, y,
    n_estimators=100,
    max_depth=20,
    model_save_path=None
):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        n_jobs=-1,
        random_state=42
    )

    model.fit(X, y)

    # Save model if path provided
    if model_save_path:
        joblib.dump(model, model_save_path)
        print(f"Model saved to {model_save_path}")

    return model

In [ ]:
def train_rf_with_validation(
    X, y,
    n_estimators_list=[50, 100, 150],
    max_depth=20,
    test_size=0.2,
    random_state=42,
    model_save_path=None
):
    """
    Train Random Forest with validation split and select best model.
    Simulates early stopping by picking best n_estimators.

    Returns:
    - best_model
    """

    # Split data
    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    best_model = None
    best_auc = -1
    best_n = None

    for n in n_estimators_list:
        print(f"\nTraining with n_estimators={n}")

        model = RandomForestClassifier(
            n_estimators=n,
            max_depth=max_depth,
            n_jobs=-1,
            random_state=random_state,
            class_weight="balanced"
        )

        model.fit(X_train, y_train)

        # Validation performance
        y_val_prob = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_val_prob)

        print(f"Validation AUC: {auc:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_model = model
            best_n = n

    print(f"\nBest model: n_estimators={best_n}, AUC={best_auc:.4f}")

    # Save model if path provided
    if model_save_path:
        joblib.dump(best_model, model_save_path)
        print(f"Model saved to {model_save_path}")

    return best_model

In [ ]:
def evaluate(model, X, y, name="Model"):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    print(f"\n{name}")
    print("-" * 50)
    print("AUC:", roc_auc_score(y, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y, y_pred))
    print(classification_report(y, y_pred))

In [ ]:
def plot_feature_importance(model, top_n=20):
    importances = model.feature_importances_
    idx = np.argsort(importances)[-top_n:]

    plt.figure(figsize=(8,6))
    plt.barh(range(top_n), importances[idx])
    plt.yticks(range(top_n), idx)
    plt.title("Top Feature Importance")
    plt.show()

## 5. Data Paths

In [ ]:
pdf_train_chunks = [
    "/content/drive/MyDrive/CSCE-704/Project/ember_pdf_train_train_0.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_pdf_train_train_1.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_pdf_train_train_2.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_pdf_train_train_3.npz"
]

elf_train_chunks = [
    "/content/drive/MyDrive/CSCE-704/Project/ember_elf_train_train_0.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_elf_train_train_1.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_elf_train_train_2.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_elf_train_train_3.npz"
]

apk_train_chunks = [
    "/content/drive/MyDrive/CSCE-704/Project/ember_apk_train_train_0.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_apk_train_train_1.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_apk_train_train_2.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_apk_train_train_3.npz"
]

dotnet_train_chunks = [
    "/content/drive/MyDrive/CSCE-704/Project/dotnet_train_0.npz",
    "/content/drive/MyDrive/CSCE-704/Project/dotnet_train_1.npz",
    "/content/drive/MyDrive/CSCE-704/Project/dotnet_train_2.npz",
    "/content/drive/MyDrive/CSCE-704/Project/dotnet_train_3.npz",
    "/content/drive/MyDrive/CSCE-704/Project/dotnet_train_4.npz",
    "/content/drive/MyDrive/CSCE-704/Project/dotnet_train_5.npz"
]

win32_train_chunks = [
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_0.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_1.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_2.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_3.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_4.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_5.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_6.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_7.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_8.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_9.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_10.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_11.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_12.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_13.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_14.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_15.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_16.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_17.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_18.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_19.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_20.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_21.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_22.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_23.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_24.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_25.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_26.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_27.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_28.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_29.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_30.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win32_train_31.npz"
]

win64_train_chunks = [
    "/content/drive/MyDrive/CSCE-704/Project/win64_train_0.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win64_train_1.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win64_train_2.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win64_train_3.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win64_train_4.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win64_train_5.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win64_train_6.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win64_train_7.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win64_train_8.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win64_train_9.npz",
    "/content/drive/MyDrive/CSCE-704/Project/win64_train_10.npz"
]

ember_test_chunks = [
    "/content/drive/MyDrive/CSCE-704/Project/ember_pdf_test.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_elf_test.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_apk_test.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_dotnet_test.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_win32_test.npz",
    "/content/drive/MyDrive/CSCE-704/Project/ember_win64_test.npz"
]

## 6. Per-Filetype Model Training

### 6.1 PDF

In [ ]:
X_pdf, y_pdf = load_chunks(pdf_train_chunks)

rf_pdf = train_rf_with_validation(
    X_pdf,
    y_pdf,
    n_estimators_list=[50, 100, 150],
    max_depth=15,
    model_save_path="/content/drive/MyDrive/CSCE-704/Project/Models/rf_pdf.pkl"
)

pdf_test = np.load(ember_test_chunks[0])
X_pdf_test, y_pdf_test = pdf_test["X"], pdf_test["y"]

evaluate(rf_pdf, X_pdf_test, y_pdf_test, "PDF Model on PDF Test")

plot_feature_importance(rf_pdf)

### 6.2 ELF

In [ ]:
X_elf, y_elf = load_chunks(elf_train_chunks)

rf_elf = train_rf_with_validation(
    X_elf,
    y_elf,
    n_estimators_list=[50, 100, 150],
    max_depth=15,
    model_save_path="/content/drive/MyDrive/CSCE-704/Project/Models/rf_elf.pkl"
)

elf_test = np.load(ember_test_chunks[1])
X_elf_test, y_elf_test = elf_test["X"], elf_test["y"]

evaluate(rf_elf, X_elf_test, y_elf_test, "ELF Model on ELF Test")

plot_feature_importance(rf_elf)

### 6.3 APK

In [ ]:
X_apk, y_apk = load_chunks(apk_train_chunks)

rf_apk = train_rf_with_validation(
    X_apk,
    y_apk,
    n_estimators_list=[50, 100, 150],
    max_depth=20,
    model_save_path="/content/drive/MyDrive/CSCE-704/Project/Models/rf_apk.pkl"
)

apk_test = np.load(ember_test_chunks[2])
X_apk_test, y_apk_test = apk_test["X"], apk_test["y"]

evaluate(rf_apk, X_apk_test, y_apk_test, "APK Model on APK Test")

plot_feature_importance(rf_apk)

### 6.4 .NET

In [ ]:
X_dotnet, y_dotnet = load_chunks(dotnet_train_chunks)

rf_dotnet = train_rf_with_validation(
    X_dotnet,
    y_dotnet,
    n_estimators_list=[50, 100, 150],
    max_depth=20,
    model_save_path="/content/drive/MyDrive/CSCE-704/Project/Models/rf_dotnet.pkl"
)

dotnet_test = np.load(ember_test_chunks[3])
X_dotnet_test, y_dotnet_test = dotnet_test["X"], dotnet_test["y"]

evaluate(rf_dotnet, X_dotnet_test, y_dotnet_test, ".NET Model on .NET Test")

plot_feature_importance(rf_dotnet)

### 6.5 Win32 (Part 1 of 2)

In [ ]:
X_win32_1, y_win32_1 = load_chunks(win32_train_chunks[:16])

print("Loading Complete")

rf_win32_1 = train_rf(
    X_win32_1,
    y_win32_1,
    n_estimators=100,
    max_depth=15,
    model_save_path="/content/drive/MyDrive/CSCE-704/Project/Models/rf_win32_1.pkl"
)

win32_1_test = np.load(ember_test_chunks[4])
X_win32_1_test, y_win32_1_test = win32_1_test["X"], win32_1_test["y"]

evaluate(rf_win32_1, X_win32_1_test, y_win32_1_test, "Win-32_1 Model on Win 32 Test")

plot_feature_importance(rf_win32_1)

### 6.6 Win32 (Part 2 of 2)

In [ ]:
X_win32_2, y_win32_2 = load_chunks(win32_train_chunks[16:])

print("Loading Complete")

rf_win32_2 = train_rf(
    X_win32_2,
    y_win32_2,
    n_estimators=100,
    max_depth=15,
    model_save_path="/content/drive/MyDrive/CSCE-704/Project/Models/rf_win32_2.pkl"
)

win32_2_test = np.load(ember_test_chunks[4])
X_win32_2_test, y_win32_2_test = win32_2_test["X"], win32_2_test["y"]

evaluate(rf_win32_2, X_win32_2_test, y_win32_2_test, "Win-32_2 Model on Win-32 Test")

plot_feature_importance(rf_win32_2)

### 6.7 Win64

In [ ]:
X_win64_1, y_win64_1 = load_chunks(win64_train_chunks)

rf_win64_1 = train_rf(
    X_win64_1,
    y_win64_1,
    n_estimators=100,
    max_depth=15,
    model_save_path="/content/drive/MyDrive/CSCE-704/Project/Models/rf_win64_1.pkl"
)

win64_1_test = np.load(ember_test_chunks[5])
X_win64_1_test, y_win64_1_test = win64_1_test["X"], win64_1_test["y"]

evaluate(rf_win64_1, X_win64_1_test, y_win64_1_test, "Win-64_1 Model on Win-64 Test")

plot_feature_importance(rf_win64_1)

## 7. Global Ensemble Training

In [ ]:
pdf_sel = pdf_train_chunks[:4]
elf_sel = elf_train_chunks[:4]
apk_sel = apk_train_chunks[:4]
dotnet_sel = dotnet_train_chunks[:6]
win32_sel = win32_train_chunks[21:26]
win64_sel = win64_train_chunks[5:10]

In [ ]:
X_pdf, y_pdf = load_chunks(pdf_sel)
X_elf, y_elf = load_chunks(elf_sel)
X_apk, y_apk = load_chunks(apk_sel)
X_dotnet, y_dotnet = load_chunks(dotnet_sel)
X_win32, y_win32 = load_chunks(win32_sel)
X_win64, y_win64 = load_chunks(win64_sel)

In [ ]:
X_global = np.vstack([
    X_pdf, X_elf, X_apk,
    X_dotnet, X_win32, X_win64
])

y_global = np.hstack([
    y_pdf, y_elf, y_apk,
    y_dotnet, y_win32, y_win64
])

print(X_global.shape)

In [ ]:
rf_global = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced"
)

rf_global.fit(X_global, y_global)

joblib.dump(rf_global, "/content/drive/MyDrive/CSCE-704/Project/Models/rf_global_5.pkl")

print("Global model saved successfully.")

## 8. Global Ensemble Evaluation (Test Set)

In [ ]:
import joblib

model_paths = [
    "/content/drive/MyDrive/CSCE-704/Project/Models/rf_global.pkl",
    "/content/drive/MyDrive/CSCE-704/Project/Models/rf_global_1.pkl",
    "/content/drive/MyDrive/CSCE-704/Project/Models/rf_global_2.pkl",
    "/content/drive/MyDrive/CSCE-704/Project/Models/rf_global_3.pkl",
    "/content/drive/MyDrive/CSCE-704/Project/Models/rf_global_4.pkl",
    "/content/drive/MyDrive/CSCE-704/Project/Models/rf_global_5.pkl"
]

models = [joblib.load(p) for p in model_paths]

print(f"Loaded {len(models)} models")

In [ ]:
def combine_tests(paths):
    X_list, y_list = [], []
    for p in paths:
        data = np.load(p)
        X_list.append(data["X"])
        y_list.append(data["y"])
    return np.vstack(X_list), np.hstack(y_list)


X_test, y_test = combine_tests(ember_test_chunks)

def ensemble_predict_proba(models, X):
    probs = np.zeros(X.shape[0])

    for model in models:
        probs += model.predict_proba(X)[:, 1]

    probs /= len(models)
    return probs

y_prob = ensemble_predict_proba(models, X_test)
y_pred = (y_prob > 0.5).astype(int)

print("\nEnsemble Global Model")
print("-" * 50)
print("AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

## 9. Challenge Set Evaluation

In [ ]:
import thrember

%mkdir /content/ember_challenge_data

thrember.download_dataset("/content/ember_challenge_data", split="challenge")

In [ ]:
import os
import json
import numpy as np
import thrember

# Correct extractor initialization
extractor = thrember.PEFeatureExtractor()

def vectorize_challenge(data_dir):
    X_list = []

    files = sorted([
        f for f in os.listdir(data_dir)
        if f.endswith(".jsonl")
    ])

    for file in files:
        print(f"Processing {file}")

        path = os.path.join(data_dir, file)

        with open(path, "r") as f:
            for line in f:
                row = json.loads(line)

                try:
                    vec = extractor.process_raw_features(row)
                    X_list.append(vec)

                except Exception:
                    continue

    X = np.array(X_list, dtype=np.float32)

    print(f"\nFinal challenge shape: {X.shape}")

    return X

X_challenge = vectorize_challenge("/content/ember_challenge_data")

y_prob = ensemble_predict_proba(models, X_challenge)

y_pred = (y_prob > 0.5).astype(int)

detection_rate = np.mean(y_pred == 1)

print("Challenge Detection Rate:", detection_rate)


print("Mean malicious confidence:", np.mean(y_prob))
print("Median malicious confidence:", np.median(y_prob))

In [ ]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

for t in thresholds:
    y_pred = (y_prob > t).astype(int)
    detection_rate = np.mean(y_pred == 1)

    print(f"Threshold {t:.1f} -> Detection Rate: {detection_rate:.4f}")